# Preprocessing

<div style="text-align: justify">
The following notebook is dedicated to preprocessing for the  <b> SUSY tau+X </b> search analysis. Preprocessing involves handling .ROOT files for use in both cut-and-count- and machine learning-based analyses. The general workflow is structured as follows:

</div>

<img src="../assets/workflow_preprocessing.png" alt="Workflow" style="width: 60%; display: block; margin: 0 auto;"/>

## Initialization

### Libraries

Data I/O:
* [UpROOT](https://uproot.readthedocs.io/en/latest/)
* [tqdm](https://tqdm.github.io/)

Data Processing:
* [Awkward Array](https://awkward-array.readthedocs.io/en/latest/)
* [NumPy](https://numpy.org/)

Lifecycle Management:
* [pickle](https://docs.python.org/3/library/pickle.html)

### Notebook

Activating autoreload of imported modules.

In [1]:
%load_ext autoreload
%autoreload 2

Initializing the project root.

In [2]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

In [3]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

Supressing unessential warnings.

In [4]:
suppress_warnings()

Unessential warnings suppressed.


Applying ATLAS style and LaTeX compatibility.

In [5]:
apply_atlas_style()

Notice: LaTeX not found on this system. Applying ATLAS style without LaTeX.


## Definition

List of available analyses.

<br>

<hr>

Run 3:
* <b> release </b>: R24
* <b> analysis base </b>: '24.2.28'
* <b> campaigns </b>: ['MC23a', 'MC23c']
* <b> architecture </b>: 'RNN'
* <b> datas </b>: ['data']
* <b> backgrounds </b>: ['ttbar', 'wtaunu', 'wmunu', 'wenu', 'ztautau', 'zmumu', 'zee', 'znunu', 'diboson', 'singletop', 'ttX']
* <b> signals </b>: gluinos -> ['GG_XXX_YYY'] & squarks -> ['SS_XXX_YYY']

<br>

<hr>

Run 2:
* <b> release </b>: 'R24'
* <b> analysis base </b>: '24.2.28'
* <b> campaigns </b>: ['MC20a', 'MC20d', 'MC20e']
* <b> architecture </b>: 'RNN'
* <b> datas </b>: ['data']
* <b> backgrounds </b>: ['ttbar', 'wtaunu', 'wmunu', 'wenu', 'ztautau', 'zmumu', 'zee', 'znunu', 'diboson', 'higgs', 'singletop', 'ttX']
* <b> signals </b>: gluinos -> ['GG_XXX_YYY'] & squarks -> ['SS_XXX_YYY']

<br>

<hr>

Defining the analysis of interest.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config")

In [ ]:
from src.processing.analysis import AnalysisConfig, resolve_samples

Defining the analysis of interest.

In [ ]:
analysis = AnalysisConfig(**cfg.analysis)

Determining the analysis data.

In [8]:
samples = resolve_samples(cfg)

In [9]:
analysis

AnalysisConfig(run='Run2', release='R24', analysis_base='24.2.28', campaigns=['MC20a', 'MC20d', 'MC20e'], scope='ML', region='Preselection', channel='1', subject=None, sub_subject=None)

In [10]:
samples

{'data': [Sample(id='data', label='data')],
 'background': [Sample(id='ttbar', label='$t \\bar{t}$'),
  Sample(id='wtaunu', label='$W \\rightarrow \\tau\\nu$'),
  Sample(id='wmunu', label='$W \\rightarrow \\mu\\nu$'),
  Sample(id='wenu', label='$W \\rightarrow e \\nu$'),
  Sample(id='ztautau', label='$Z \\rightarrow \\tau\\tau$'),
  Sample(id='zmumu', label='$Z \\rightarrow \\mu\\mu$'),
  Sample(id='zee', label='$Z \\rightarrow ee$'),
  Sample(id='znunu', label='$Z \\rightarrow \\nu\\nu$'),
  Sample(id='diboson', label='diboson'),
  Sample(id='singletop', label='singletop')],
 'fake': [],
 'signal': []}

In [16]:
bg_ids    = [s.id for s in samples["background"]]
fake_ids  = [s.id for s in samples["fake"]]
signal_ids = [s.id for s in samples["signal"]]

Process backgrounds

In [ ]:
from src.processing.processor import process_samples
bg_raw = process_samples(cfg, sample_type="background", sample_ids=bg_ids)

Process fakes

In [21]:
fake_raw = process_samples(cfg, sample_type="fake", sample_ids=fake_ids) if fake_ids else {}

In [ ]:
signal_raw = process_samples(cfg, sample_type="signal", sample_ids=signal_ids) if signal_ids else {}

In [ ]:
# ── Cell 8: Merge ──
from src.processing.merger import (
    merge_backgrounds,
    merge_fakes,
    combine_background_fake,
    combine_background_signal,
    assign_class,
)

bg_merged = merge_backgrounds(bg_raw, cfg)

if fake_raw:
    fake_merged = merge_fakes(fake_raw)
    bg_merged = combine_background_fake(bg_merged, fake_merged)

if signal_raw:
    all_samples = combine_background_signal(bg_merged, signal_raw)
else:
    all_samples = bg_merged

assign_class(all_samples)

In [ ]:
# ── Cell 9: Save to parquet ──
from src.processing.io import save_samples

output_dir = path / "data" / "processed" / cfg.analysis.run.lower()
save_samples(all_samples, output_dir)

Each cell maps to one pipeline step:

1. Autoreload — hot-reloads modules as you edit them
2. Root setup — adds project to sys.path
3. Hydra config — loads config.yaml with features/paths/merge defaults
4. Resolve samples — reads sample configs, applies excludes, discovers signals
5. Process backgrounds — reads ROOT, applies all cuts, computes weights per
campaign
6. Process fakes — same pipeline with fake-specific truth-matching and ff_weights
7. Process signals — same pipeline, no truth cuts
8. Merge — groups backgrounds (as_primary → topquarks/wtaunu/ztautau/diboson),
merges fakes into faketaus, combines, assigns class labels
9. Save — writes one parquet file per key to data/processed/run2/